# Unit 11 - AI System Evaluation (Demo) · **V2 material**

**Atoms served:** `U11-A1` (non-determinism), `U11-A2` (subjective `OEC`), `U11-A3` (`LLM`-as-judge), `U11-A4` (binding guardrails), `U11-A5` (model drift)

**Estimated runtime:** ~25 seconds · **No API keys** - everything is simulated with a fixed seed.

**After this notebook you can:** say which experiment decisions break under non-deterministic treatments, score judge disagreement, spot judge bias, read a quality-cost frontier, and see drift hide inside a pooled estimate.

## Without code

This small table is **illustrative** - it shows the shape of the data rather than the exact rows the code below generates:

| id | arm | output length | judge A | judge B |
|----|-----|---------------|---------|---------|
| 1 | control | 40 | 3.2 | 3.0 |
| 2 | treatment | 120 | 4.5 | 3.8 |
| 3 | treatment | 35 | 3.1 | 3.4 |
| 4 | control | 38 | 3.0 | 2.9 |

1. **Non-determinism:** same arm, different outputs (rows 2 vs 3) - variance inflates required `n`.
2. **Judge disagreement:** Cohen's kappa between A and B comes out around **0.12** - barely better than chance, not "moderate." Two plausible judges are nowhere near interchangeable.
3. **Bias:** judge A scores correlate with length; a length-blind human would shrink the treatment effect.
4. **Guardrails:** treatment wins on quality but doubles cost - frontier point (quality=4.2, cost=2.0) vs control (3.0, 1.0).
5. **Drift:** after the midpoint, treatment quality drops 0.8 points. The pooled `ATE` still reads about **+0.2**, which looks like a small win; split by period it is **+0.76** early and **-0.30** late.

**Note:** There is no dedicated video for this unit - the notebook is often the first place you meet these ideas.

## 1. The question

You want to A/B test a new model-backed answer against a fixed template. The output varies every call, quality is subjective, cost matters, and the vendor may update the model mid-test. Which parts of a standard experiment plan still work?

## 2. Setup

**Before you run:** the install line in the next cell is commented out on purpose - Colab already has every library this notebook needs. If an import fails, remove the `#` and run the cell again.

In [ ]:
# Colab already ships numpy, pandas, scipy, statsmodels and matplotlib, so the
# install line below stays commented out and this cell runs instantly.
# If any import in this cell fails, delete the leading # and run the cell again.
# %pip install -q numpy pandas scipy statsmodels matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

RANDOM_SEED = 42  # change this and re-run to see how luck changes the story
np.random.seed(RANDOM_SEED)
plt.rcParams['figure.figsize'] = (7, 4)

## 3. The data

Simulate model outputs as random draws - no external API. Control = low-variance template; treatment = higher mean but noisier responses.

### 1. Non-determinism (`U11-A1`)

Draw multiple outputs per arm. Variance under treatment should exceed control, inflating the sample size you need for the same power.

In [ ]:
def simulate_output(arm, rng):
    if arm == 0:
        return rng.normal(3.0, 0.2)
    return rng.normal(3.5, 0.8)

rng = np.random.default_rng(RANDOM_SEED)
ctrl = [simulate_output(0, rng) for _ in range(500)]
trt = [simulate_output(1, rng) for _ in range(500)]
var_ctrl = np.var(ctrl, ddof=1)
var_trt = np.var(trt, ddof=1)
print('Variance control:', round(var_ctrl, 3), 'treatment:', round(var_trt, 3))
print('Variance ratio (treatment/control):', round(var_trt / var_ctrl, 1))

Higher variance under treatment means you need more samples to see the same lift - `SUTVA`'s consistency condition broke first.

## 4. The naive move

Treat one model score as ground truth and pool all days into one t-test.

### 2. Two judges and disagreement (`U11-A2`, `U11-A3`)

Two judges score the same outputs. Report disagreement with Cohen's kappa (`V24` inter-rater frame).

In [ ]:
n_q = 80
quality = rng.normal(3.2, 0.6, n_q)
length = rng.integers(30, 150, n_q)
judge_a = quality + 0.01 * length + rng.normal(0, 0.3, n_q)
judge_b = quality + rng.normal(0, 0.5, n_q)

# bucket into low/mid/high for kappa
def bucket(x):
    return pd.cut(x, [-np.inf, 2.8, 3.6, np.inf], labels=[0, 1, 2]).codes

ka = bucket(judge_a)
kb = bucket(judge_b)
agree = (ka == kb).mean()
# simple kappa
po = agree
pe = sum((ka==k).mean() * (kb==k).mean() for k in [0,1,2])
kappa = (po - pe) / (1 - pe) if pe < 1 else 0
print('Judge agreement rate:', round(po, 3), 'Cohen kappa:', round(kappa, 3))

A kappa this low means "whose judgment" is not settled at all - the two judges agree barely more than they would by chance. Define your raters and measure their agreement **before** you promote any judge score to an `OEC`.

## 5. What actually happens

### 3. Biased judge (`U11-A3`)

Judge A rewards length. Estimate treatment effect on judge A vs debiased scores (quality without length penalty).

In [ ]:
arms = rng.integers(0, 2, 200)
base_q = 3.0 + 0.4 * arms + rng.normal(0, 0.3, 200)
out_len = 50 + 40 * arms + rng.integers(-10, 10, 200)
score_a = base_q + 0.02 * out_len + rng.normal(0, 0.2, 200)
score_debiased = base_q  # human blind to length
ate_a = score_a[arms==1].mean() - score_a[arms==0].mean()
ate_true = score_debiased[arms==1].mean() - score_debiased[arms==0].mean()
print('Biased judge ATE:', round(ate_a, 3))
print('Debiased ATE:', round(ate_true, 3))

The biased judge inflates the treatment effect because treatment answers are longer. Check agreement with humans; report bias direction.

### 4. Guardrails that bind (`U11-A4`)

Compare arms on quality and cost per request. Plot the frontier.

In [ ]:
options = pd.DataFrame({
    'arm': ['control', 'treatment A', 'treatment B'],
    'quality': [3.0, 4.2, 3.8],
    'cost_per_request': [1.0, 2.0, 1.2]
})
fig, ax = plt.subplots()
ax.scatter(options['cost_per_request'], options['quality'])
for _, r in options.iterrows():
    ax.annotate(r['arm'], (r['cost_per_request'], r['quality']))
ax.set_xlabel('cost per request')
ax.set_ylabel('quality score')
ax.set_title('Better quality often costs more - guardrails decide')
plt.show()

Treatment A wins on quality but fails a 1.5x cost guardrail - the guardrail is the decision, not decoration.

### 5. Drift (`U11-A5`)

Mid-experiment, the treatment distribution drops by 0.8. Pooled analysis misses the shift; period-split analysis catches it.

In [ ]:
days = 20
mid = days // 2
treat_early = rng.normal(3.6, 0.4, mid)
treat_late = rng.normal(2.8, 0.4, days - mid)  # drift after model update
ctrl_all = rng.normal(3.0, 0.3, days)
treat_all = np.concatenate([treat_early, treat_late])
pooled_ate = treat_all.mean() - ctrl_all.mean()
early_ate = treat_early.mean() - ctrl_all[:mid].mean()
late_ate = treat_late.mean() - ctrl_all[mid:].mean()
print('Pooled ATE:', round(pooled_ate, 3))
print('Early period ATE:', round(early_ate, 3), 'Late period ATE:', round(late_ate, 3))

A pooled `ATE` of about +0.2 reads as a modest win and hides the actual story: a real early gain that reversed into a loss after the model changed under you. Nothing in the pooled number is false - it is just the average of two different experiments. This is instrumentation drift from `V22`, and there is still no standard dashboard alert for it.

## 6. What you do about it

- Pin or cache treatments when you can; budget extra `n` for output variance (`U11-A1`).
- Define **whose judgment** and measure **inter-rater agreement** before the `OEC` (`U11-A2`, `U11-A3`).
- Treat **cost and latency** as binding guardrails, not background metrics (`U11-A4`).
- Split by period or pin model version to catch **drift** (`U11-A5`).

**When this matters less:** Deterministic, cheap, objectively scored treatments - your old playbook still fits.

---

**Takeaway:** Non-determinism, subjective scoring, binding guardrails, and drift stress the Eight Decisions you already learned - they do not replace them.

**Back to the unit:** [V2 unit 11](../V2/units/unit-11-experimenting-with-ai-systems/README.md)